# Style2Fit — Step 1: Generate Training Data
Converts Fashion-Gen dataset into casual situation → structured outfit pairs for LLM fine-tuning.

**Runtime:** CPU is fine. Takes ~20 min for 500 pairs.

In [ ]:
!pip install anthropic datasets -q

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-...'  # paste your key here

In [ ]:
import json
import random
import re
from pathlib import Path
import anthropic
from datasets import load_dataset

random.seed(42)
Path('data').mkdir(exist_ok=True)

client = anthropic.Anthropic()

REQUIRED_FIELDS = ['Top:', 'Bottom:', 'Shoes:', 'Aesthetic:', 'Explanation:']

CATEGORY_SIGNALS = {
    'top':       ['shirt', 'tee', 'blouse', 'sweater', 'top', 'jacket', 'coat',
                  'blazer', 'cardigan', 'hoodie', 'sweatshirt', 'vest', 'tank'],
    'bottom':    ['pant', 'trouser', 'jean', 'skirt', 'short', 'legging', 'chino',
                  'slack', 'culotte', 'wide-leg', 'straight-leg'],
    'shoes':     ['shoe', 'boot', 'sneaker', 'heel', 'loafer', 'sandal', 'flat',
                  'oxford', 'pump', 'mule', 'wedge'],
    'outerwear': ['coat', 'jacket', 'blazer', 'trench', 'parka', 'anorak',
                  'overcoat', 'windbreaker', 'cape'],
    'accessory': ['bag', 'belt', 'scarf', 'hat', 'earring', 'necklace', 'bracelet',
                  'sunglasses', 'watch', 'purse', 'tote', 'clutch'],
}

SYSTEM_PROMPT = """You are generating training data for a fashion AI assistant called Style2Fit.

Given a fashion product description, output TWO things:

1. SITUATION: A casual, conversational prompt a real person might type — like texting a friend.
   - Vary the register: sometimes lowercase, sometimes with typos, sometimes incomplete
   - The situation should naturally lead someone to want this outfit
   - Examples: "coffee date tmrw help", "first day at my internship", "going to a rooftop bar friday"

2. OUTFIT: The structured outfit plan derived from the product description.
   - Fill in any missing pieces logically
   - Keep descriptions specific: color, material, fit where available
   - Aesthetic should be 1-2 words
   - Explanation should be 2 sentences max, conversational

Always respond in EXACTLY this format, no extra text:
SITUATION: <casual prompt>
---
Top: <item>
Bottom: <item>
Shoes: <item>
Outerwear: <item or "none needed">
Accessories: <2-3 items>
Aesthetic: <1-2 words>
Explanation: <why this works>"""

print('Setup complete.')

In [ ]:
def is_full_outfit(entry):
    description = entry.get('description') or entry.get('caption') or ''
    if len(description) < 40:
        return False
    desc_lower = description.lower()
    categories_found = sum(
        1 for signals in CATEGORY_SIGNALS.values()
        if any(s in desc_lower for s in signals)
    )
    return categories_found >= 3


def convert_entry(description):
    try:
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=400,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': f'Fashion description:\n{description}'}],
        )
        text = response.content[0].text.strip()

        if '---' not in text or 'SITUATION:' not in text:
            return None

        parts = text.split('---', 1)
        situation = re.sub(r'^SITUATION:\s*', '', parts[0].strip(), flags=re.IGNORECASE).strip()
        outfit_text = parts[1].strip()

        if not all(f in outfit_text for f in REQUIRED_FIELDS):
            return None

        return {
            'instruction': situation,
            'input': '',
            'output': outfit_text,
            'source': 'fashiongen',
        }
    except Exception as e:
        print(f'  Error: {e}')
        return None


print('Functions defined.')

In [ ]:
N = 500  # number of training pairs to generate

print('Loading Fashion-Gen...')
ds = load_dataset('rajistics/fashion-gen', split='train')
print(f'  Total entries: {len(ds)}')

print('Filtering to full outfits...')
full_outfits = [e for e in ds if is_full_outfit(e)]
print(f'  Full outfit entries: {len(full_outfits)} ({len(full_outfits)/len(ds)*100:.1f}%)')

sampled = random.sample(full_outfits, min(N * 2, len(full_outfits)))
print(f'  Sampled {len(sampled)} to convert (2x target to account for failures)')

In [ ]:
pairs = []
attempts = 0

print(f'Converting to {N} training pairs...')
for entry in sampled:
    if len(pairs) >= N:
        break
    description = entry.get('description') or entry.get('caption') or ''
    pair = convert_entry(description)
    attempts += 1
    if pair:
        pairs.append(pair)
        if len(pairs) % 50 == 0:
            print(f'  {len(pairs)}/{N} done (attempts: {attempts})')

print(f'\nSuccess rate: {len(pairs)}/{attempts} ({len(pairs)/max(attempts,1)*100:.1f}%)')

In [ ]:
out_path = 'data/train.jsonl'
with open(out_path, 'w') as f:
    for pair in pairs:
        f.write(json.dumps(pair) + '\n')
print(f'Saved {len(pairs)} pairs to {out_path}')

# Preview first example
print('\n--- Sample pair ---')
print(f"SITUATION: {pairs[0]['instruction']}")
print(f"OUTFIT:\n{pairs[0]['output']}")

In [ ]:
# Download the file to your machine
from google.colab import files
files.download('data/train.jsonl')